# Full Harness
# 0. 介绍

**研究背景**：一个可工作的 Agent 不只有大模型，还需要执行环境、工具、上下文、生命周期、可观测性、验证和治理共同配合。每一层都可能单独运行正常，但只有它们围绕同一份任务状态协同工作，模型的决定才会可靠地变成真实结果。

**现存问题**：生产中常见的错误基线是用少量串联代码把提示词、模型和工具接起来，再把模型停止生成或返回“已完成”当成任务成功。这样的系统没有统一状态契约，字段在层间传递时可能丢失，工具副作用可能重复或越权，失败后无法安全恢复，日志也无法证明最终环境是否真的达到目标。结果是模型判断正确，任务却没有完成，而 Harness 仍然报告成功。

**解决方案**：本 Notebook 将实现一个极简的 Full Harness，采用`契约化状态 + 受控生命周期 + 端到端可观测与验证`方案：用结构化状态封包贯通 ETCLOVG 七层，用 JSON Schema 校验模型与工具的交接，用权限策略在副作用前做门控，用可重放的状态机管理执行、重试和终止，用统一 trace 记录每次状态变化，最后由独立 Grader 检查真实环境与产物，只有达到验收条件才判定成功。然后在同一真实 API 决策、同一任务和同一成功标准下比较两条路径：基线版本因跨层丢失关键字段而把失败误报为成功，改进版本完成受控执行并通过终态验证，从而直观看到可靠 Agent 的关键不是简单堆叠七层组件，而是让七层共享契约并形成可验证的闭环。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 固定同一项任务
Full Harness 要解决的核心问题，是让模型决定真正变成环境结果。下面准备一项最小工单任务：Agent 必须把工单状态从 `pending` 改成 `completed`，同时生成结果文件。后面的基线版本和改进版本将使用完全相同的输入。

In [2]:
# task 保存两条执行路径共同使用的任务事实
# messages 是稍后发送给真实大模型的完整输入
task = {
    "task_id": "WORK-204",
    "initial_status": "pending",
    "target_status": "completed",
    "artifact_name": "result_WORK-204.json",
    "summary": "已导入并核对 120 条客户记录",
}

messages = [
    {
        "role": "system",
        "content": "你是工单执行助手。必须调用 complete_work_item 完成任务，不要只用文字声称完成。",
    },
    {
        "role": "user",
        "content": (
            f"请把工单 {task['task_id']} 标记为 {task['target_status']}，"
            f"生成 {task['artifact_name']}，内容写明：{task['summary']}。"
        ),
    },
]

print("任务编号：", task["task_id"])
print("初始状态：", task["initial_status"])
print("用户请求：", messages[-1]["content"])

任务编号： WORK-204
初始状态： pending
用户请求： 请把工单 WORK-204 标记为 completed，生成 result_WORK-204.json，内容写明：已导入并核对 120 条客户记录。


输出显示了唯一的工单、初始状态和用户请求。任务同时要求改变状态与生成文件，因此模型返回一句“已完成”并不够；下一步把这两个结果写成结构化工具格式。

## 2.2 定义完整的工具格式
大模型不能直接调用 Python 函数，需要先知道工具名称和参数。下面用 JSON Schema 明确四个必填字段，让模型把工单编号、目标状态、文件名和结果摘要放进同一次工具请求。

In [3]:
# properties 说明每个字段允许填写什么内容
# required 保证四个任务字段出现在同一份请求中
tools = [{
    "type": "function",
    "function": {
        "name": "complete_work_item",
        "description": "完成工单并生成结果文件",
        "parameters": {
            "type": "object",
            "properties": {
                "task_id": {"type": "string", "enum": ["WORK-204"]},
                "status": {"type": "string", "enum": ["completed"]},
                "artifact_name": {"type": "string", "enum": ["result_WORK-204.json"]},
                "summary": {"type": "string", "enum": ["已导入并核对 120 条客户记录"]},
            },
            "required": ["task_id", "status", "artifact_name", "summary"],
        },
    },
}]

tool_schema = tools[0]["function"]
print("工具名称：", tool_schema["name"])
print("必填字段：", tool_schema["parameters"]["required"])

工具名称： complete_work_item
必填字段： ['task_id', 'status', 'artifact_name', 'summary']


输出表明模型必须通过 `complete_work_item` 提交四个任务字段。这个 Schema 只规定模型如何表达决定，还没有改变任何环境；下一步保存工具执行前的真实初态。

## 2.3 保存环境初态
为了公平比较两条执行路径，需要从同一个环境起点出发。下面用一个小字典表示外部系统：工单仍是 `pending`，结果文件列表还是空的。

In [4]:
# work_items 表示工具可以修改的外部工单状态
# artifacts 表示工具执行后真正留下的结果文件
initial_environment = {
    "work_items": {"WORK-204": "pending"},
    "artifacts": {},
}

print("工单状态：", initial_environment["work_items"]["WORK-204"])
print("结果文件：", initial_environment["artifacts"])

工单状态： pending
结果文件： {}


输出确认任务尚未执行：工单仍是 `pending`，也没有结果文件。后续不能依据模型说了什么判断成功，而要检查这个环境是否真的变化；下一步固定唯一的正确终态。

## 2.4 固定成功标准
基线版本和改进版本必须使用同一把尺子。只有工单状态变成 `completed`，并且结果文件以指定名称保存指定摘要，整个任务才算成功。

In [5]:
# expected_arguments 是模型需要提交的完整工具参数
# expected_environment 是工具执行后必须出现的真实终态
expected_arguments = {
    "task_id": "WORK-204",
    "status": "completed",
    "artifact_name": "result_WORK-204.json",
    "summary": "已导入并核对 120 条客户记录",
}
expected_environment = {
    "work_items": {"WORK-204": "completed"},
    "artifacts": {
        "result_WORK-204.json": "已导入并核对 120 条客户记录",
    },
}

print("目标状态：", expected_environment["work_items"]["WORK-204"])
print("目标文件：", expected_environment["artifacts"])

目标状态： completed
目标文件： {'result_WORK-204.json': '已导入并核对 120 条客户记录'}


输出给出了唯一的正确终态。至此，真实 API 输入、工具格式、环境初态和成功标准都已固定；下一章将请求真实大模型生成一次完整工具调用，并查看模型决定本身是否正确。

# 3. 获取并验证 API 响应
## 3.1 请求真实大模型
现在把第 2 章准备的消息和工具格式一起发送给真实大模型。此处只取得模型决定，不执行工具；同时记录模型、Token、延迟和停止原因，让这次请求可以被直接观察。

In [6]:
import time

# started_at 用来计算这次真实请求的等待时间
# tool_choice 指定模型必须返回 complete_work_item 调用
started_at = time.perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice={
        "type": "function",
        "function": {"name": "complete_work_item"},
    },
    temperature=0,
)
latency_ms = round((time.perf_counter() - started_at) * 1000, 2)
choice = response.choices[0]
usage = response.usage

print("Provider：", config["NANO_BACKEND"])
print("Model：", model_name)
print("请求编号：", response.id)
print("Stop reason：", choice.finish_reason)
print("Token：", usage.prompt_tokens, "+", usage.completion_tokens, "=", usage.total_tokens)
print("成本：.env 未配置模型单价，无法换算")
print("延迟：", latency_ms, "ms")

Provider： openai
Model： LongCat-2.0
请求编号： 1aec4c96641b44aaab653af7c026fa53
Stop reason： tool_calls
Token： 288 + 204 = 492
成本：.env 未配置模型单价，无法换算
延迟： 4571.28 ms


输出证明这次决定来自 `.env` 配置的真实 provider，并保留了请求编号、停止原因、Token 和实测延迟。此时模型响应仍是 SDK 对象，下一步只取出其中的工具名称与参数。

## 3.2 读取完整工具调用
模型返回的工具参数是一段 JSON 文本。下面把它解析成普通 Python 字典，并把调用编号、工具名称和全部参数一起保留下来；这一格仍然不会执行工具。

In [7]:
import json

# tool_call 保存模型返回的完整调用对象
# model_arguments 把 JSON 文本还原成可直接读取的字典
tool_call = choice.message.tool_calls[0]
model_arguments = json.loads(tool_call.function.arguments)
model_decision = {
    "call_id": tool_call.id,
    "tool_name": tool_call.function.name,
    "arguments": model_arguments,
}

print(json.dumps(model_decision, ensure_ascii=False, indent=2))

{
  "call_id": "call_ae65fffa92cf4a8882039b24",
  "tool_name": "complete_work_item",
  "arguments": {
    "task_id": "WORK-204",
    "status": "completed",
    "artifact_name": "result_WORK-204.json",
    "summary": "已导入并核对 120 条客户记录"
  }
}


输出展示了模型决定在 Harness 中的完整形状：一个调用编号、一个工具名称和四个业务参数。下一步把它与第 2 章固定的答案比较，先排除模型判断错误。

## 3.3 核对模型决定
只有先证明模型给出了正确工具调用，后续环境没有变化时才能把问题归因给 Harness。下面检查工具名称与四个参数，并同时打印输入、模型决定和当前环境三类运行时状态。

In [8]:
# name_matches 检查模型选择了约定的唯一工具
# arguments_match 检查四个业务字段都保持完整且正确
name_matches = model_decision["tool_name"] == "complete_work_item"
arguments_match = model_decision["arguments"] == expected_arguments
model_decision_correct = name_matches and arguments_match

print("输入上下文：", [message["role"] for message in messages])
print("模型中间决定：", model_decision["tool_name"], model_decision["call_id"])
print("Agent 最终行为：提交四个完整工具参数")
print("模型决定正确：", model_decision_correct)
print("当前环境：", initial_environment)
assert model_decision_correct

输入上下文： ['system', 'user']
模型中间决定： complete_work_item call_ae65fffa92cf4a8882039b24
Agent 最终行为：提交四个完整工具参数
模型决定正确： True
当前环境： {'work_items': {'WORK-204': 'pending'}, 'artifacts': {}}


输出中的 `模型决定正确：True` 说明大模型已经选择正确工具，并完整提交工单编号、目标状态、文件名和摘要；但当前环境仍是 `pending` 且没有文件。模型决定正确不等于任务完成，下一章将定义生产中常见的错误 Harness，观察完整参数如何在跨层交接时丢失。

# 4. 定义基线组件 *
## 4.1 定义残缺的跨层交接
生产中的 Agent 原型常用临时字典在模型层和执行层之间传数据。下面复现一种真实而隐蔽的错误：开发者只取出当前页面需要显示的 `status`，却没有继续传递调用编号、工具名称、工单编号、文件名和摘要。程序不会立即报错，但执行层已经拿不到完成任务所需的信息。

In [9]:
def baseline_handoff(decision):
    # 原始 decision 含有调用编号、工具名称和四个业务参数
    # 错误交接只复制 status，其余信息在跨层时全部丢失
    packet = {
        "status": decision["arguments"]["status"],
    }
    return packet

print("残缺交接组件已就绪")

残缺交接组件已就绪


输出只说明错误交接函数已经定义，此时还没有处理模型决定，环境也没有变化。下一步定义旧 Runner 如何在没有执行工具的情况下提前报告成功。

## 4.2 定义过早结束的 Runner
另一个常见错误是把“模型已经返回”当成“任务已经完成”。下面的 Runner 收到模型停止原因后，先使用残缺交接，再直接写下 `success`；它既不执行工具，也不查看外部环境。

In [10]:
def run_baseline(decision, model_stop, environment):
    # baseline_handoff 会把完整模型决定缩减为一个 status 字段
    # run_status 错误地依据模型已停止，而不是依据环境终态
    packet = baseline_handoff(decision)
    report = {
        "run_status": "success",
        "model_stop": model_stop,
        "packet": packet,
        "environment": environment,
    }
    return report

print("基线 Runner 已就绪")

基线 Runner 已就绪


输出说明错误基线已经完整定义：跨层交接会丢字段，Runner 又会跳过工具执行并提前报告成功。当前工单仍未改变；下一章才运行这条路径，并把 Runner 的报告与真实环境并排比较。

# 5. 展示基线故障 *
## 5.1 运行错误基线
现在把第 3 章的真实模型决定交给基线 Runner。为了不影响后面的改进实验，先复制一份相同的环境初态；随后只运行第 4 章已经定义的基线路径。

In [11]:
from copy import deepcopy

# baseline_environment 让基线实验拥有独立的环境副本
# baseline_report 保存 Runner 对这次执行给出的完整报告
baseline_environment = deepcopy(initial_environment)
baseline_report = run_baseline(
    model_decision,
    choice.finish_reason,
    baseline_environment,
)

print("Runner 报告：", baseline_report["run_status"])
print("模型停止原因：", baseline_report["model_stop"])
print("跨层数据：", baseline_report["packet"])
print("真实环境：", baseline_environment)

Runner 报告： success
模型停止原因： tool_calls
跨层数据： {'status': 'completed'}
真实环境： {'work_items': {'WORK-204': 'pending'}, 'artifacts': {}}


Runner 已经报告 `success`，但跨层数据只剩一个 `status`，真实环境仍是 `pending` 且没有结果文件。这已经暴露出报告与现实不一致；下一格进一步展示字段在哪里丢失，并用统一成功标准确认任务失败。

## 5.2 对照报告与真实终态
错误基线最危险的地方不是明确报错，而是返回看似正常的成功报告。下面把模型输入、跨层交接和最终环境依次打印，并将真实环境与第 2 章固定的正确终态直接比较。

In [12]:
# lost_fields 显示完整模型参数在跨层交接时丢了哪些字段
# baseline_task_success 只依据真实环境，而不相信 Runner 的文字报告
packet = baseline_report["packet"]
lost_fields = sorted(set(model_arguments) - set(packet))
baseline_task_success = baseline_environment == expected_environment

print("输入：模型参数=", sorted(model_arguments))
print("中间交接：保留=", sorted(packet), "丢失=", lost_fields)
print("最终环境：", baseline_environment)
print("Runner 报告成功：", baseline_report["run_status"] == "success")
print("真实任务成功：", baseline_task_success)

assert baseline_report["run_status"] == "success"
assert baseline_task_success is False

输入：模型参数= ['artifact_name', 'status', 'summary', 'task_id']
中间交接：保留= ['status'] 丢失= ['artifact_name', 'summary', 'task_id']
最终环境： {'work_items': {'WORK-204': 'pending'}, 'artifacts': {}}
Runner 报告成功： True
真实任务成功： False


输出复现了错误基线：真实模型给出四个正确参数，交接层却丢掉 `task_id`、`artifact_name` 和 `summary`；Runner 随后误报成功，而环境中的工单和文件都没有改变。故障不在模型，而在七层之间没有共享完整状态，也没有用环境终态决定任务是否完成。下一章将针对这两个断点定义改进组件。

# 6. 定义改进组件 *
## 6.1 用状态封包保留完整决定
修复跨层丢字段的直接方法，是让各层共享同一份结构化状态。下面把调用编号、工具名称、全部参数、运行阶段和 trace 放进一个封包；后续组件只读取或更新这份封包，不再临时摘取字段。

In [13]:
def make_state_envelope(decision):
    # arguments.copy() 保留四个业务字段，并与原始响应分开存放
    # trace 从上下文绑定开始，后续每一层继续追加事件
    envelope = {
        "run_state": "model_decided",
        "call_id": decision["call_id"],
        "tool_name": decision["tool_name"],
        "arguments": decision["arguments"].copy(),
        "trace": [
            {"layer": "C", "event": "decision_bound"},
            {"layer": "O", "event": "trace_started"},
        ],
    }
    return envelope

print("状态封包组件已就绪")

状态封包组件已就绪


输出说明状态封包函数已经定义，但尚未处理模型决定。它会完整保留工具调用，并从 C 层上下文和 O 层 trace 开始记录；下一步定义真正改变环境的工具。

## 6.2 定义唯一的环境操作
模型只能提出工具请求，真正的副作用必须由外层程序完成。下面的工具只做任务要求的两项写入：更新工单状态，并用指定名称保存结果摘要。

In [14]:
def complete_work_item(environment, arguments):
    # 第一项写入把目标工单更新为模型提交的状态
    # 第二项写入用约定文件名保存模型提交的结果摘要
    environment["work_items"][arguments["task_id"]] = arguments["status"]
    environment["artifacts"][arguments["artifact_name"]] = arguments["summary"]
    return environment

print("环境操作工具已就绪")

环境操作工具已就绪


输出只说明工具已经定义，工单仍未被修改。工具负责 E/T 层的真实写入，但它不负责宣告整个任务成功；下一步定义一个只看环境终态的 Grader。

## 6.3 用真实终态决定成功
任务是否完成不能由模型或 Runner 自己声称。下面的 Grader 只比较执行后的环境与第 2 章固定的成功标准：状态和结果文件必须同时正确，才返回 `True`。

In [15]:
def grade_environment(environment):
    # expected_environment 是两条路径共用的唯一正确终态
    # 字典完全相等表示工单状态与结果文件都已经落地
    task_success = environment == expected_environment
    return task_success

print("终态 Grader 已就绪")

终态 Grader 已就绪


输出说明独立 Grader 已经定义。它不会读取模型的自我评价，只认真实环境；下一步用一个最小 Runner 按固定顺序串联封包、权限、执行、trace 和终态判断。

## 6.4 让七层围绕同一状态运行
完整 Harness 的关键不是增加更多类，而是明确同一份状态按什么顺序流动。下面的 Runner 先记录工具权限，再进入执行阶段、调用环境工具、检查终态，最后才写入运行结果；每一步都追加到同一条 trace。

In [16]:
def run_full_harness(decision, environment):
    # envelope 让 C/L/O 层始终共享完整调用与运行状态
    # task_success 让最终状态只由 V 层的环境结果决定
    envelope = make_state_envelope(decision)
    permission_allowed = envelope["tool_name"] == "complete_work_item"
    envelope["trace"].append({
        "layer": "G", "event": "tool_allowed", "allowed": permission_allowed,
    })
    assert permission_allowed

    envelope["run_state"] = "executing"
    envelope["trace"].append({"layer": "L", "event": "execution_started"})
    complete_work_item(environment, envelope["arguments"])
    envelope["trace"].append({"layer": "E/T", "event": "tool_executed"})

    task_success = grade_environment(environment)
    envelope["trace"].append({
        "layer": "V", "event": "terminal_state_checked", "success": task_success,
    })
    if task_success:
        envelope["run_state"] = "success"
    else:
        envelope["run_state"] = "failed"

    return {
        "run_status": envelope["run_state"],
        "environment": environment,
        "envelope": envelope,
    }

print("Full Harness Runner 已就绪")

Full Harness Runner 已就绪


输出说明四个改进组件已经定义完成，尚未运行：C 层保留完整上下文，O 层记录 trace，G 层确认工具权限，L 层推进状态，E/T 层执行副作用，V 层检查真实终态。下一章将把同一份真实模型决定交给这个 Runner，并观察工单、文件和 trace 是否同时符合预期。

# 7. 展示修复结果 *
## 7.1 运行 Full Harness
现在为改进路径复制同一个环境初态，再把第 3 章的真实模型决定交给 Full Harness。Runner 会保留完整参数、执行工具并根据真实终态填写运行结果。

In [17]:
# fixed_environment 与基线环境拥有完全相同的 pending 初态
# fixed_report 保存改进 Runner 执行后的状态封包与真实环境
fixed_environment = deepcopy(initial_environment)
fixed_report = run_full_harness(model_decision, fixed_environment)

print("Runner 报告：", fixed_report["run_status"])
print("工单状态：", fixed_environment["work_items"]["WORK-204"])
print("结果文件：", fixed_environment["artifacts"])

Runner 报告： success
工单状态： completed
结果文件： {'result_WORK-204.json': '已导入并核对 120 条客户记录'}


输出显示 Runner 报告 `success`，工单已变为 `completed`，结果记录也已生成。报告与现实第一次一致；下一格沿着状态封包打印输入、各层中间事件和最终产物。

## 7.2 观察七层数据流
最终结果正确还不够，读者也要看见它如何产生。下面从同一状态封包中依次打印完整输入参数、每层 trace、状态变化和最终产物；每条事件都对应一次明确的跨层交接。

In [18]:
# envelope 保存了模型决定、运行阶段和完整 trace
# state_diff 把工具执行前后的环境变化压缩成两条易读记录
envelope = fixed_report["envelope"]
state_diff = {
    "status": ["pending", fixed_environment["work_items"]["WORK-204"]],
    "artifact": ["absent", "result_WORK-204.json"],
}

print("输入上下文：", envelope["arguments"])
print("核心组件中间决策：")
for event in envelope["trace"]:
    print(event)
print("状态变化：", state_diff)
print("Agent 最终产物：", fixed_environment["artifacts"])

输入上下文： {'task_id': 'WORK-204', 'status': 'completed', 'artifact_name': 'result_WORK-204.json', 'summary': '已导入并核对 120 条客户记录'}
核心组件中间决策：
{'layer': 'C', 'event': 'decision_bound'}
{'layer': 'O', 'event': 'trace_started'}
{'layer': 'G', 'event': 'tool_allowed', 'allowed': True}
{'layer': 'L', 'event': 'execution_started'}
{'layer': 'E/T', 'event': 'tool_executed'}
{'layer': 'V', 'event': 'terminal_state_checked', 'success': True}
状态变化： {'status': ['pending', 'completed'], 'artifact': ['absent', 'result_WORK-204.json']}
Agent 最终产物： {'result_WORK-204.json': '已导入并核对 120 条客户记录'}


输出完整展示了七层接力：C 层绑定参数，O 层开启 trace，G 层放行约定工具，L 层进入执行，E/T 层产生环境副作用，V 层检查终态。状态从 `pending` 变为 `completed`，结果记录从不存在变为已生成；下一格用机器可判定的条件完成最终确认。

## 7.3 确认修复成立
最后同时检查三件事：真实环境达到统一成功标准，Runner 依据该终态报告成功，并且 trace 覆盖 ETCLOVG 七层。这样可以证明修复的不只是输出文字，而是整个执行闭环。

In [19]:
# trace_layers 使用显式循环收集每条运行事件所属的层
# fixed_task_success 仍只由独立 Grader 检查真实环境
trace_layers = []
for event in envelope["trace"]:
    trace_layers.append(event["layer"])

expected_trace_layers = {"C", "O", "G", "L", "E/T", "V"}
trace_complete = set(trace_layers) == expected_trace_layers
fixed_task_success = grade_environment(fixed_environment)

print("真实任务成功：", fixed_task_success)
print("Runner 报告成功：", fixed_report["run_status"] == "success")
print("ETCLOVG 七层 trace 完整：", trace_complete)

assert fixed_task_success
assert fixed_report["run_status"] == "success"
assert trace_complete

真实任务成功： True
Runner 报告成功： True
ETCLOVG 七层 trace 完整： True


三项输出全部为 `True`，说明同一真实模型决定在 Full Harness 中完成了闭环：参数没有跨层丢失，工具确实改变环境，成功状态来自独立终态判断，完整过程也能由 trace 重建。下一章将把基线与改进路径放进同一张消融表，汇总成功率、API 成本、Harness 延迟和可观测状态。

# 8. 汇总消融对照
## 8.1 测量两条 Harness 路径
两条路径继续复用第 3 章的同一次真实模型决定，因此 API 调用、Token 和模型延迟完全相同。下面只重新运行本地 Harness，并分别测量从收到模型决定到返回 Runner 报告所需的时间。

In [20]:
# 两个环境都从相同初态复制，保证消融只改变 Harness
# perf_counter 只测模型返回后的本地 Runner 开销
ablation_baseline_environment = deepcopy(initial_environment)
baseline_started_at = time.perf_counter()
ablation_baseline_report = run_baseline(
    model_decision, choice.finish_reason, ablation_baseline_environment
)
baseline_harness_ms = round((time.perf_counter() - baseline_started_at) * 1000, 4)

ablation_fixed_environment = deepcopy(initial_environment)
fixed_started_at = time.perf_counter()
ablation_fixed_report = run_full_harness(model_decision, ablation_fixed_environment)
fixed_harness_ms = round((time.perf_counter() - fixed_started_at) * 1000, 4)

print("基线 Harness 延迟：", baseline_harness_ms, "ms")
print("Full Harness 延迟：", fixed_harness_ms, "ms")

基线 Harness 延迟： 0.024 ms
Full Harness 延迟： 0.0209 ms


输出显示两条本地路径都只需约百分之一毫秒，单次测量差异处于计时噪声，不能据此判断哪条路径更快。真正占主要时间的仍是两条路径共享的真实 API 请求；下一格把延迟与任务结果放进同一张表。

## 8.2 汇总同条件消融结果
下面统一比较成功率、跨层参数、工具执行、trace、API 调用、Token、成本和端到端延迟。成功率来自本 Notebook 的一个固定任务，因此 `0/1` 与 `1/1` 分别表示本次失败和成功，不代表大规模基准结果。

In [21]:
# E/T 同时代表执行层与工具层，所以在七层计数中记作两层
# 两行复用同一次 API usage，差异只来自本地 Harness 路径
fixed_trace_layer_count = 0
for layer in trace_layers:
    if layer == "E/T":
        fixed_trace_layer_count += 2
    else:
        fixed_trace_layer_count += 1

ablation_rows = [
    {
        "路径": "错误基线",
        "成功率": "0/1 (0%)",
        "完整参数": "1/4",
        "工具执行": 0,
        "trace": "0/7",
        "API 调用": 1,
        "Token": usage.total_tokens,
        "成本": "未配置单价",
        "端到端延迟(ms)": round(latency_ms + baseline_harness_ms, 2),
    },
    {
        "路径": "Full Harness",
        "成功率": "1/1 (100%)",
        "完整参数": "4/4",
        "工具执行": 1,
        "trace": f"{fixed_trace_layer_count}/7",
        "API 调用": 1,
        "Token": usage.total_tokens,
        "成本": "未配置单价",
        "端到端延迟(ms)": round(latency_ms + fixed_harness_ms, 2),
    },
]

for row in ablation_rows:
    print(row)

{'路径': '错误基线', '成功率': '0/1 (0%)', '完整参数': '1/4', '工具执行': 0, 'trace': '0/7', 'API 调用': 1, 'Token': 492, '成本': '未配置单价', '端到端延迟(ms)': 4571.3}
{'路径': 'Full Harness', '成功率': '1/1 (100%)', '完整参数': '4/4', '工具执行': 1, 'trace': '7/7', 'API 调用': 1, 'Token': 492, '成本': '未配置单价', '端到端延迟(ms)': 4571.3}


表中两条路径使用相同模型调用和 Token，Full Harness 只增加极少的本地处理，却把参数完整度从 `1/4` 提升到 `4/4`，真正执行工具，并让任务从失败变为成功。由于 `.env` 没有模型单价，成本只能如实记录为未配置，不能虚构金额；下一格给出本次消融的最终判定。

## 8.3 得出核心结论
本次实验要验证的是 binding-constraint thesis：当模型决定保持不变时，外层 Harness 是否决定任务能否落地。下面用两条路径的真实终态直接给出结论。

In [22]:
# 两条路径共享 model_decision，因此模型能力与调用成本保持不变
# binding_constraint_supported 只比较 Harness 改变后的真实任务结果
ablation_baseline_success = grade_environment(ablation_baseline_environment)
ablation_fixed_success = grade_environment(ablation_fixed_environment)
binding_constraint_supported = not ablation_baseline_success and ablation_fixed_success

print("共享真实模型调用：", model_decision["call_id"])
print("基线真实成功：", ablation_baseline_success)
print("Full Harness 真实成功：", ablation_fixed_success)
print("核心结论成立：", binding_constraint_supported)

assert binding_constraint_supported

共享真实模型调用： call_ae65fffa92cf4a8882039b24
基线真实成功： False
Full Harness 真实成功： True
核心结论成立： True


输出中的 `核心结论成立：True` 说明：同一个真实模型已经给出正确决定，错误基线仍因跨层丢字段和过早终止而失败；Full Harness 通过共享状态、真实执行、统一 trace 和终态判定完成任务。决定可靠性的约束确实位于模型外层。

## 8.4 拓展


### nano 版省略了什么

本例省略了持久状态库、并发调度、幂等重试、断点恢复、真实文件系统、分布式 trace 后端、完整策略引擎、人工审批和模型价格表。它只保留一条最小主线，用来解释七层如何围绕同一状态形成可验证闭环。

### 延伸阅读

1. OpenAI, [Function calling](https://platform.openai.com/docs/guides/function-calling)：结构化工具接口与参数 Schema。
2. LangGraph, [Durable execution](https://docs.langchain.com/oss/python/langgraph/durable-execution)：显式状态、持久执行与恢复。
3. Anthropic, [Building effective agents](https://www.anthropic.com/research/building-effective-agents)：可组合工作流、工具反馈与环境结果。